# Colab — Build features (pipeline clásico)

Este notebook ejecuta la etapa más lenta del pipeline clásico (Selective Search + HOG + SIFT + BoVW sobre cada imagen train) **en Colab**, no en tu PC.

Salida: `classical_features.npz` (~150 MB) que luego se entrena en `colab_train_svm.ipynb`.

**Soporta checkpoint**: si Colab desconecta, vuelve a abrir el notebook y dale Run All — reanuda desde donde quedó.

## Prerequisitos en Drive

Antes de la primera ejecución, sube **manualmente** estos archivos a Google Drive en `/MyDrive/grocery-detection/`:

```
/MyDrive/grocery-detection/
    raw/
        d2s_images_v*.tar.xz         ← descargado de MVTec (acepta licencia)
        d2s_annotations_v*.tar.xz
    processed/
        train.json                   ← output local de scripts/prepare_splits.py
        val.json
        test.json
        codebook.pkl                 ← output local de scripts/train_codebook.py
```

`prepare_splits.py` y `train_codebook.py` son baratos (1-5 min en CPU); cómodos en local. Si tu PC tampoco aguanta eso, hay celdas opcionales más abajo para generarlos en Colab.

## 1. Bootstrap — clona repo + importa helpers

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    sync_from_drive, sync_to_drive, run_script,
)
print("helpers cargados")

## 2. Montar Drive + instalar deps

In [ ]:
mount_drive()
install_deps()

## 3. Extraer D2S al disco local

Copia los .tar.xz de Drive a /content y los extrae. ~5-10 min la primera vez por sesión (re-runs en la misma sesión saltan).

In [ ]:
setup_dataset()

## 4. Bajar splits + codebook + checkpoint previo desde Drive

Si es tu primera ejecución, `classical_features.npz` aún no existe en Drive → se salta (mensaje skip).

In [ ]:
sync_from_drive([
    "train.json",
    "val.json",
    "test.json",
    "codebook.pkl",
    "classical_features.npz",
])

## 5. Build features (con checkpoint cada 100 imgs)

Esto es lo que tarda horas. `--skip-svm-fit` evita entrenar la SVM aquí (eso lo hacemos en `colab_train_svm.ipynb` con sample_steps=2 + n_jobs=-1).

In [ ]:
run_script("scripts/run_classical_train.py", "--skip-svm-fit")

## 6. Subir features.npz de vuelta a Drive

**Importante**: si Colab cortó a mitad, el checkpoint parcial ya se fue subiendo periódicamente vía esta celda — pero si no llegaste a esta celda, no se subió. Mientras tanto, el .npz local tiene los datos. Re-ejecuta solo esta celda si fuese necesario.

In [ ]:
sync_to_drive(["classical_features.npz"])

## Siguiente paso

Abre `notebooks/colab_train_svm.ipynb` en Colab. Subirá `classical_features.npz` (desde Drive ahora también vale) y entrenará la SVM en minutos.

## Opcional — generar splits + codebook aquí mismo

Si tu PC va MUY lento incluso para los pasos baratos, descomenta y corre estas celdas (no necesarias si ya subiste `train.json`/`val.json`/`test.json`/`codebook.pkl` desde local).

In [ ]:
# Descomentar si quieres generar los splits en Colab:
# run_script("scripts/prepare_splits.py")
# sync_to_drive(["train.json", "val.json", "test.json"])

In [ ]:
# Descomentar si quieres entrenar el codebook en Colab:
# run_script("scripts/train_codebook.py")
# sync_to_drive(["codebook.pkl"])